## Data

In [ ]:
# This notebook tests the FrogDQ algorithm by comparing five scenarios:
# 1.  **Clean Baseline**: A standard MLP trained on the original dataset.
# 2.  **Partial Noise Baseline**: A standard MLP trained on the original dataset where a fraction of features have been corrupted by Gaussian noise.
# 3.  **Partial Noise + FrogDQ**: A FrogDQ-enabled MLP trained on the partially corrupted dataset.
# 4.  **Appended Noise Baseline**: A standard MLP trained on the dataset augmented with completely random, useless features.
# 5.  **Appended Noise + FrogDQ**: A FrogDQ-enabled MLP trained on the augmented dataset with appended noise.
#
# The goal is to plot the training curves to analyze model performance under different types of feature noise.

# %%
# =========================
# Config (edit as needed)
# =========================
dataset_name             = "adult"       # "adult", "mushroom", "breast_cancer", "iris"
# --- Appended Noise Config ---
N_noise                  = 50            # Number of random columns to add
noise_quality            = 0.1           # q for appended noise features (0..1)
# --- Partial Noise Config ---
PARTIAL_NOISE_FRACTION   = 0.5           # Fraction of original features to corrupt
PARTIAL_NOISE_SCALE      = 0.5           # Std deviation of noise to add
PARTIAL_NOISE_Q          = 0.1           # q for partially noisy features (0..1)
# --- General Training Config ---
val_size                 = 0.2
test_size                = 0.2
BASE_SEED                = 42            # Seeds dataset creation + everything else
NUM_RUNS                 = 30
EPOCHS                   = 300
BATCH_SIZE               = 128
LR                       = 1e-3
HIDDEN_DIM               = 128
DROPOUT                  = 0.1
LAMBDA_PROX              = 0.1           # FrogDQ proximal loss strength
EPS                      = 1e-8
# --- Early Stopping Config ---
EARLY_STOP                = True
ES_PATIENCE               = 30        # epochs with no improvement before stopping
ES_MIN_DELTA              = 1e-4      # minimum improvement to reset patience
ES_MONITOR                = "val_acc" # maximize this metric


# =========================
# Imports
# =========================
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_openml, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# =========================
# Reproducibility
# =========================
def set_seed(seed: int = 42):
    """Sets the seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # The following two lines are needed for full determinism with CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(BASE_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Data Loading and Preparation
# =========================
def load_dataset(name: str):
    """Loads a specified dataset from sklearn or OpenML."""
    name = name.lower()
    if name == "adult":
        df = fetch_openml(name="adult", version=2, as_frame=True, parser='auto')
        X = df.data
        y = (df.target.astype(str).str.contains(">")).astype(int).values
    elif name == "mushroom":
        df = fetch_openml(name="mushroom", version=1, as_frame=True, parser='auto')
        X = df.data
        y = (df.target.astype(str) == "p").astype(int).values  # p=poisonous
    elif name == "breast_cancer":
        ds = load_breast_cancer(as_frame=True)
        X = ds.frame.drop(columns=["target"])
        y = ds.target.values
    elif name == "iris":
        ds = load_iris(as_frame=True)
        X = ds.frame.drop(columns=["target"])
        y = ds.target.values
    else:
        raise ValueError(f"Unknown dataset '{name}'")
    return X, y

# Load raw data
X_raw, y_raw = load_dataset(dataset_name)

# Identify column types
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns
num_cols = X_raw.select_dtypes(include=np.number).columns

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

# Stratified splits (Train-Val / Test, then Train / Val)
X_train_val, X_test_df, y_train_val, y_test = train_test_split(
    X_raw, y_raw, test_size=test_size, stratify=y_raw, random_state=BASE_SEED
)
X_train_df, X_val_df, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=(val_size / (1.0 - test_size)), # Adjust val_size for the second split
    stratify=y_train_val, random_state=BASE_SEED
)

# Fit preprocessor on training data and transform all sets
preprocessor.fit(X_train_df)
X_train_clean = preprocessor.transform(X_train_df)
X_val_clean = preprocessor.transform(X_val_df)
X_test_clean = preprocessor.transform(X_test_df)
original_dim = X_train_clean.shape[1]

# Create reproducible random number generator for noise
rng_noise = np.random.RandomState(BASE_SEED)

# --- Create "Partial Noise" dataset and its q-vector ---
n_features_to_corrupt = int(original_dim * PARTIAL_NOISE_FRACTION)
feature_indices_to_corrupt = rng_noise.choice(original_dim, n_features_to_corrupt, replace=False)

def add_partial_noise(X, indices, scale, rng):
    X_noisy = X.copy()
    noise_to_add = rng.randn(X.shape[0], len(indices)) * scale
    X_noisy[:, indices] += noise_to_add
    return X_noisy

X_train_partial = add_partial_noise(X_train_clean, feature_indices_to_corrupt, PARTIAL_NOISE_SCALE, rng_noise)
X_val_partial = add_partial_noise(X_val_clean, feature_indices_to_corrupt, PARTIAL_NOISE_SCALE, rng_noise)
X_test_partial = add_partial_noise(X_test_clean, feature_indices_to_corrupt, PARTIAL_NOISE_SCALE, rng_noise)

# q-vector for partial noise: 1 for clean features, PARTIAL_NOISE_Q for corrupted ones
q_vec_partial = np.ones(original_dim, dtype=np.float32)
q_vec_partial[feature_indices_to_corrupt] = PARTIAL_NOISE_Q


# --- Create "Appended Noise" dataset and its q-vector ---
noise_train = rng_noise.randn(X_train_clean.shape[0], N_noise)
noise_val = rng_noise.randn(X_val_clean.shape[0], N_noise)
noise_test = rng_noise.randn(X_test_clean.shape[0], N_noise)

X_train_appended = np.hstack([X_train_clean, noise_train])
X_val_appended = np.hstack([X_val_clean, noise_val])
X_test_appended = np.hstack([X_test_clean, noise_test])

# q-vector for appended noise: 1 for original, `noise_quality` for new ones
q_original = np.ones(original_dim, dtype=np.float32)
q_noise_features = np.full(N_noise, noise_quality, dtype=np.float32)
q_vec_appended = np.concatenate([q_original, q_noise_features])

# Convert all data to PyTorch tensors
X_train_clean_t = torch.tensor(X_train_clean, dtype=torch.float32)
X_val_clean_t = torch.tensor(X_val_clean, dtype=torch.float32)
X_test_clean_t = torch.tensor(X_test_clean, dtype=torch.float32)

X_train_partial_t = torch.tensor(X_train_partial, dtype=torch.float32)
X_val_partial_t = torch.tensor(X_val_partial, dtype=torch.float32)
X_test_partial_t = torch.tensor(X_test_partial, dtype=torch.float32)

X_train_appended_t = torch.tensor(X_train_appended, dtype=torch.float32)
X_val_appended_t = torch.tensor(X_val_appended, dtype=torch.float32)
X_test_appended_t = torch.tensor(X_test_appended, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

q_vec_partial_t = torch.tensor(q_vec_partial, dtype=torch.float32, device=device)
q_vec_appended_t = torch.tensor(q_vec_appended, dtype=torch.float32, device=device)

# Determine number of classes from training data
NUM_CLASSES = len(np.unique(y_train))

print(f"--- Experiment Setup ---")
print(f"Dataset: {dataset_name.capitalize()}")
print(f"Train/Val/Test split: {len(y_train)}/{len(y_val)}/{len(y_test)}")
print(f"Feature dimensions: Original={original_dim}, Appended Noise={N_noise}, Total Appended={X_train_appended.shape[1]}")
print(f"Partial noise corrupts {n_features_to_corrupt}/{original_dim} features with scale {PARTIAL_NOISE_SCALE} and q={PARTIAL_NOISE_Q}.")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Device: {device}")
print(f"------------------------\n")

# %%
# =========================
# Models
# =========================
class MLP(nn.Module):
    """Standard Multi-Layer Perceptron."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

class FrogGate(nn.Module):
    """A learnable gate for each input feature."""
    def __init__(self, input_dim):
        super().__init__()
        self.gates = nn.Parameter(torch.ones(input_dim, dtype=torch.float32))
    def forward(self, x):
        return x * self.gates

class FrogMLP(nn.Module):
    """MLP with FrogDQ input gates."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout):
        super().__init__()
        self.gate = FrogGate(input_dim)
        self.body = MLP(input_dim, hidden_dim, output_dim, dropout)
    def forward(self, x):
        return self.body(self.gate(x))

# %%
# =========================
# Unified Training and Evaluation
# =========================
def train(model, X_train, y_train, X_val, y_val, **kwargs):
    """Unified training loop for both standard and FrogDQ models, with optional early stopping."""
    # Hyperparameters
    epochs = kwargs.get("epochs", EPOCHS)
    batch_size = kwargs.get("batch_size", BATCH_SIZE)
    lr = kwargs.get("lr", LR)
    seed = kwargs.get("seed", BASE_SEED)

    # Early stopping options
    use_es = kwargs.get("early_stop", EARLY_STOP)
    es_patience = kwargs.get("es_patience", ES_PATIENCE)
    es_min_delta = kwargs.get("es_min_delta", ES_MIN_DELTA)
    es_monitor = kwargs.get("es_monitor", ES_MONITOR)  # "val_acc" (maximize)

    # FrogDQ specific params
    is_frogdq = 'q_vec' in kwargs
    if is_frogdq:
        q_vec = kwargs['q_vec']
        lambda_prox = kwargs.get('lambda_prox', LAMBDA_PROX)
        # Proximal weights from q: low q => high inertia weight (1-q)
        w = 1.0 - q_vec
        w = w / (w.mean() + EPS)

    set_seed(seed)
    model.to(device)

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_acc": [], "epochs_run": 0}

    # Initialize previous gates for proximal loss
    if is_frogdq:
        g_prev = model.gate.gates.detach().clone()

    # Early-stopping bookkeeping
    best_metric = -float("inf")  # higher is better for val_acc
    best_state = None
    patience_left = es_patience
    stopped_early = False

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss_ce = criterion(logits, yb)
            loss = loss_ce

            if is_frogdq:
                g_curr = model.gate.gates
                loss_prox = lambda_prox * (((g_curr - g_prev) ** 2) * w).sum()
                loss = loss + loss_prox

            loss.backward()
            optimizer.step()

            if is_frogdq:
                g_prev = g_curr.detach().clone()

            total_train_loss += loss_ce.item() * xb.size(0)

        # Log training loss
        history["train_loss"].append(total_train_loss / len(train_ds))

        # Validation accuracy
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                preds = logits.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

        val_acc = correct / total
        history["val_acc"].append(val_acc)

        # ---- Early stopping check (maximize val_acc) ----
        current_metric = val_acc if es_monitor == "val_acc" else val_acc  # easy extension
        if current_metric > best_metric + es_min_delta:
            best_metric = current_metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = es_patience
        else:
            patience_left -= 1
            if use_es and patience_left <= 0:
                stopped_early = True
                history["epochs_run"] = epoch + 1
                break

    # If never set (e.g., no early stop), record full epoch count
    if history["epochs_run"] == 0:
        history["epochs_run"] = len(history["val_acc"])

    # Restore the best weights if we have them
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return history

def evaluate_test(model, X_test, y_test):
    """Evaluates the model on the test set."""
    model.to(device)
    model.eval()
    ds = TensorDataset(X_test, y_test)
    loader = DataLoader(ds, batch_size=512, shuffle=False)
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

# %%
# =========================
# Experiment Runner
# =========================
def init_logs():
    return {"val_acc": [], "train_loss": [], "test_acc": [], "stop_epoch": []}

logs = {
    "Clean": init_logs(),
    "Partial_Standard": init_logs(),
    "Partial_FrogDQ": init_logs(),
    "Appended_Standard": init_logs(),
    "Appended_FrogDQ": init_logs()
}

for i in range(NUM_RUNS):
    run_seed = BASE_SEED + i
    print(f"--- Starting Run {i+1}/{NUM_RUNS} (seed={run_seed}) ---")

    # 1. Clean Baseline
    model_clean = MLP(original_dim, HIDDEN_DIM, NUM_CLASSES, DROPOUT)
    hist_clean = train(model_clean, X_train_clean_t, y_train_t, X_val_clean_t, y_val_t,
                       seed=run_seed, early_stop=EARLY_STOP,
                       es_patience=ES_PATIENCE, es_min_delta=ES_MIN_DELTA, es_monitor=ES_MONITOR)
    logs["Clean"]["val_acc"].append(hist_clean["val_acc"])
    logs["Clean"]["train_loss"].append(hist_clean["train_loss"])
    logs["Clean"]["stop_epoch"].append(hist_clean["epochs_run"])
    logs["Clean"]["test_acc"].append(evaluate_test(model_clean, X_test_clean_t, y_test_t))

    # 2. Partial Noise Baseline
    model_partial = MLP(original_dim, HIDDEN_DIM, NUM_CLASSES, DROPOUT)
    hist_partial = train(model_partial, X_train_partial_t, y_train_t, X_val_partial_t, y_val_t,
                         seed=run_seed, early_stop=EARLY_STOP,
                         es_patience=ES_PATIENCE, es_min_delta=ES_MIN_DELTA, es_monitor=ES_MONITOR)
    logs["Partial_Standard"]["val_acc"].append(hist_partial["val_acc"])
    logs["Partial_Standard"]["train_loss"].append(hist_partial["train_loss"])
    logs["Partial_Standard"]["stop_epoch"].append(hist_partial["epochs_run"])
    logs["Partial_Standard"]["test_acc"].append(evaluate_test(model_partial, X_test_partial_t, y_test_t))

    # 3. Partial Noise + FrogDQ
    model_partial_frog = FrogMLP(original_dim, HIDDEN_DIM, NUM_CLASSES, DROPOUT)
    hist_partial_frog = train(
        model_partial_frog, X_train_partial_t, y_train_t, X_val_partial_t, y_val_t,
        seed=run_seed, q_vec=q_vec_partial_t, lambda_prox=LAMBDA_PROX,
        early_stop=EARLY_STOP, es_patience=ES_PATIENCE, es_min_delta=ES_MIN_DELTA, es_monitor=ES_MONITOR
    )
    logs["Partial_FrogDQ"]["val_acc"].append(hist_partial_frog["val_acc"])
    logs["Partial_FrogDQ"]["train_loss"].append(hist_partial_frog["train_loss"])
    logs["Partial_FrogDQ"]["stop_epoch"].append(hist_partial_frog["epochs_run"])
    logs["Partial_FrogDQ"]["test_acc"].append(evaluate_test(model_partial_frog, X_test_partial_t, y_test_t))

    # 4. Appended Noise Baseline
    model_appended = MLP(X_train_appended.shape[1], HIDDEN_DIM, NUM_CLASSES, DROPOUT)
    hist_appended = train(model_appended, X_train_appended_t, y_train_t, X_val_appended_t, y_val_t,
                          seed=run_seed, early_stop=EARLY_STOP,
                          es_patience=ES_PATIENCE, es_min_delta=ES_MIN_DELTA, es_monitor=ES_MONITOR)
    logs["Appended_Standard"]["val_acc"].append(hist_appended["val_acc"])
    logs["Appended_Standard"]["train_loss"].append(hist_appended["train_loss"])
    logs["Appended_Standard"]["stop_epoch"].append(hist_appended["epochs_run"])
    logs["Appended_Standard"]["test_acc"].append(evaluate_test(model_appended, X_test_appended_t, y_test_t))

    # 5. Appended Noise + FrogDQ
    model_frog = FrogMLP(X_train_appended.shape[1], HIDDEN_DIM, NUM_CLASSES, DROPOUT)
    hist_frog = train(
        model_frog, X_train_appended_t, y_train_t, X_val_appended_t, y_val_t,
        seed=run_seed, q_vec=q_vec_appended_t, lambda_prox=LAMBDA_PROX,
        early_stop=EARLY_STOP, es_patience=ES_PATIENCE, es_min_delta=ES_MIN_DELTA, es_monitor=ES_MONITOR
    )
    logs["Appended_FrogDQ"]["val_acc"].append(hist_frog["val_acc"])
    logs["Appended_FrogDQ"]["train_loss"].append(hist_frog["train_loss"])
    logs["Appended_FrogDQ"]["stop_epoch"].append(hist_frog["epochs_run"])
    logs["Appended_FrogDQ"]["test_acc"].append(evaluate_test(model_frog, X_test_appended_t, y_test_t))

print("\n--- All runs completed! ---\n")

# =========================
# Aggregate and Plot Results
# =========================
def pad_curve(seq, target_len):
    arr = np.full(target_len, np.nan, dtype=np.float64)
    arr[:len(seq)] = np.asarray(seq, dtype=np.float64)
    return arr

def aggregate_curves_variable(list_of_lists, target_len):
    """Pad with NaNs, then nanmean / nanstd across runs."""
    if len(list_of_lists) == 0:
        return np.zeros(target_len), np.zeros(target_len)
    stacked = np.vstack([pad_curve(s, target_len) for s in list_of_lists])
    return np.nanmean(stacked, axis=0), np.nanstd(stacked, axis=0)

# Aggregate results for plotting
mean_acc, std_acc = {}, {}
mean_loss, std_loss = {}, {}
final_test_acc = {}
mean_stop_epoch = {}

for key in logs:
    # Maximum possible timeline is EPOCHS (padded with NaNs where runs stopped early)
    mean_acc[key], std_acc[key]   = aggregate_curves_variable(logs[key]["val_acc"], EPOCHS)
    mean_loss[key], std_loss[key] = aggregate_curves_variable(logs[key]["train_loss"], EPOCHS)
    final_test_acc[key] = (np.mean(logs[key]["test_acc"]), np.std(logs[key]["test_acc"]))
    mean_stop_epoch[key] = (np.mean(logs[key]["stop_epoch"]), np.std(logs[key]["stop_epoch"]))


# Print final test accuracies
print("--- Final Test Accuracy (Mean ± Std) ---")
print(f"Clean Baseline:           {final_test_acc['Clean'][0]:.4f} ± {final_test_acc['Clean'][1]:.4f}")
print(f"Partial Noise (MLP):      {final_test_acc['Partial_Standard'][0]:.4f} ± {final_test_acc['Partial_Standard'][1]:.4f}")
print(f"Partial Noise (FrogDQ):   {final_test_acc['Partial_FrogDQ'][0]:.4f} ± {final_test_acc['Partial_FrogDQ'][1]:.4f}")
print(f"Appended Noise (MLP):     {final_test_acc['Appended_Standard'][0]:.4f} ± {final_test_acc['Appended_Standard'][1]:.4f}")
print(f"Appended Noise (FrogDQ):  {final_test_acc['Appended_FrogDQ'][0]:.4f} ± {final_test_acc['Appended_FrogDQ'][1]:.4f}")
print("------------------------------------------\n")

# %%
# =========================
# Plot Training Curves
# =========================
epochs_range = np.arange(1, EPOCHS + 1)
plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(f'Training Curves with Early Stopping – {dataset_name.capitalize()} ({NUM_RUNS} runs)', fontsize=16)

colors = {
    'Clean': 'blue',
    'Partial_Standard': 'orange',
    'Partial_FrogDQ': 'purple',
    'Appended_Standard': 'red',
    'Appended_FrogDQ': 'green'
}
labels = {
    'Clean': 'Clean Data',
    'Partial_Standard': 'Partial Noise (MLP)',
    'Partial_FrogDQ': 'Partial Noise (FrogDQ)',
    'Appended_Standard': 'Appended Noise (MLP)',
    'Appended_FrogDQ': 'Appended Noise (FrogDQ)'
}

def plot_series(ax, x, mean_y, std_y, label, color):
    # Only plot up to the last finite (non-NaN) point
    finite_mask = np.isfinite(mean_y)
    if not np.any(finite_mask):
        return
    last_idx = np.where(finite_mask)[0][-1] + 1
    ax.plot(x[:last_idx], mean_y[:last_idx], label=label, color=color)
    ax.fill_between(
        x[:last_idx],
        (mean_y[:last_idx] - std_y[:last_idx]),
        (mean_y[:last_idx] + std_y[:last_idx]),
        alpha=0.2,
        color=color
    )

# Validation Accuracy
ax[0].set_title('Validation Accuracy vs. Epochs (mean ± std)')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
for key in logs:
    plot_series(ax[0], epochs_range, mean_acc[key], std_acc[key], labels[key], colors[key])
ax[0].legend(loc='lower right')

# Training Loss
ax[1].set_title('Training Loss vs. Epochs (mean ± std)')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Cross-Entropy Loss')
for key in logs:
    plot_series(ax[1], epochs_range, mean_loss[key], std_loss[key], labels[key], colors[key])
ax[1].set_yscale('log')
ax[1].legend(loc='upper right')

# Add vertical markers for mean early-stop epoch per scenario on both plots
for i, axi in enumerate(ax):
    for key in logs:
        m, s = mean_stop_epoch[key]
        # Draw only if mean is finite and >= 1
        if np.isfinite(m) and m >= 1:
            axi.axvline(m, linestyle='--', linewidth=0.8, color=colors[key], alpha=0.25)

plt.tight_layout(rect=[0, 0, 1, 0.94])
fig_name = f"training_curves_{dataset_name}_runs{NUM_RUNS}.png"
plt.savefig(fig_name, dpi=300, bbox_inches='tight')
plt.show()